# Centralized Q-Learning (CQL) — Demo Notebook

**Project:** Predator-Prey Archetype Gridworld  
**Algorithm:** Tabular Centralized Q-Learning  
**Audience:** Students who have completed `01_iql_demo.ipynb`

---

## What You Will Learn

1. Why IQL's independence assumption is a problem in multi-agent settings
2. What a joint state-action space is and why it scales exponentially
3. How CQL maintains a single shared Q-table across all agents
4. How marginalisation extracts per-agent action decisions from the joint Q-table
5. How CQL and IQL compare on the same environment

## Prerequisites

- Complete `01_iql_demo.ipynb` first
- Understand the Bellman update equation
- Familiarity with numpy array indexing

---
## Part 1 — The Problem with IQL

### 1.1 Non-Stationarity

In IQL, each agent maintains its own Q-table and treats the other agents as **part of the environment**. This is convenient but violates a core assumption of Q-learning:

> *Q-learning converges only if the environment is **stationary** — the same action from the same state always leads to the same distribution of outcomes.*

In a multi-agent system, the prey is **also learning**. So from the predator's perspective, the "environment" changes every episode as the prey's policy improves. This makes the effective transition dynamics non-stationary.

**Consequence:** IQL can still converge empirically on small environments, but it has no formal guarantee. In adversarial settings (predator vs. prey), the two agents can end up cycling — predator learns to go left, prey learns to go right, predator learns to compensate, etc.

### 1.2 The Centralization Solution

**Centralized Q-Learning** replaces N independent Q-tables with a **single shared Q-table** indexed by the joint state and joint action of all agents:

$$Q(\mathbf{s}, \mathbf{a}) \approx \text{expected total reward when ALL agents take actions } \mathbf{a} \text{ from joint state } \mathbf{s}$$

Where:
- $\mathbf{s} = (s_1, s_2, \ldots, s_n)$ is the tuple of all agents' individual states
- $\mathbf{a} = (a_1, a_2, \ldots, a_n)$ is the tuple of all agents' individual actions

The TD update uses the **sum of all agents' rewards** as the central signal:

$$Q(\mathbf{s}, \mathbf{a}) \leftarrow Q(\mathbf{s}, \mathbf{a}) + \alpha \Big[ \sum_i r_i + \gamma \max_{\mathbf{a}'} Q(\mathbf{s}', \mathbf{a}') - Q(\mathbf{s}, \mathbf{a}) \Big]$$

This is stationary — the joint Q-table does not change its meaning as individual policies evolve, because it reasons about all agents simultaneously.

---
## Part 2 — Setup

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from multi_agent_package.core.gridworld import GridWorldEnv
from multi_agent_package.core.agent import Agent
from multi_agent_package.registry import get_observation_builder, get_reward_function
from baselines.IQL.iql import IQL
from baselines.CQL.cql import CQL

print("All imports OK")

In [ ]:
def make_env(n_predators=1, n_prey=1, size=5, seed=42):
    """Helper to build a fresh wired environment."""
    agents = []
    for i in range(1, n_predators + 1):
        agents.append(Agent(agent_name=f"predator_{i}",
                            agent_team=f"predator_{i}", agent_type="predator"))
    for i in range(1, n_prey + 1):
        agents.append(Agent(agent_name=f"prey_{i}",
                            agent_team=f"prey_{i}", agent_type="prey"))
    
    env = GridWorldEnv(agents=agents, size=size, perc_num_obstacle=10.0,
                       render_mode=None, seed=seed,
                       capture_threshold=1, max_steps=100)
    
    obs_builder = get_observation_builder("local_radius", radius=3,
                                           include_agents=True, include_obstacles=True)
    env.observation_builder = obs_builder.build
    
    base_rf = get_reward_function("base")
    dist_rf = get_reward_function("predator_distance", weight=0.5)
    
    def combined_reward(env_inst):
        total = {ag.agent_name: 0.0 for ag in env_inst.agents}
        for rf in [base_rf, dist_rf]:
            for k, v in rf.compute(env_inst).items():
                total[k] += v
        return total
    
    env.reward_fn = combined_reward
    return env

env = make_env(n_predators=1, n_prey=1, size=5)
print(f"Environment: {env.size}×{env.size}, agents: {[ag.agent_name for ag in env.agents]}")

---
## Part 3 — The Joint State-Action Space

### 3.1 Exponential Scaling

The key cost of centralization is that the Q-table must represent every possible combination of all agents' actions. With $n$ agents each having $d$ possible actions:

$$|\mathbf{a}| = d^n$$

For this environment ($d=5$ actions):

In [ ]:
action_dim = 5  # 5 actions: Right, Up, Left, Down, Noop

print("Joint action space size for different agent counts:")
print(f"{'Agents (n)':>12s} {'Joint actions (5^n)':>22s} {'Memory per state (float32)':>28s}")
print("-" * 65)
for n in range(1, 9):
    joint = action_dim ** n
    mem_kb = joint * 4 / 1024  # float32 = 4 bytes
    flag = " ← this env (1v1)" if n == 2 else ""
    flag = " ← default 3v3"   if n == 6 else flag
    print(f"{n:>12d} {joint:>22,d} {mem_kb:>25.1f} KB{flag}")

In [ ]:
# Visualise the exponential growth
fig, ax = plt.subplots(figsize=(8, 4))
n_agents = list(range(1, 9))
joint_sizes = [action_dim ** n for n in n_agents]

ax.bar(n_agents, joint_sizes, color=["#2ecc71" if n <= 2 else
                                      "#f39c12" if n <= 4 else
                                      "#e74c3c" for n in n_agents])
ax.set_yscale("log")
ax.set_xlabel("Number of agents", fontsize=12)
ax.set_ylabel("Joint action space size (log scale)", fontsize=12)
ax.set_title("CQL State Space Explosion — 5 actions per agent", fontsize=13)
ax.set_xticks(n_agents)
ax.grid(True, alpha=0.3, axis="y")

# Annotate
for n, v in zip(n_agents, joint_sizes):
    ax.text(n, v * 1.5, f"{v:,}", ha="center", va="bottom", fontsize=7, rotation=45)

legend = [
    mpatches.Patch(color="#2ecc71", label="Tractable (≤ 2 agents)"),
    mpatches.Patch(color="#f39c12", label="Manageable (3–4 agents)"),
    mpatches.Patch(color="#e74c3c", label="Impractical (5+ agents)"),
]
ax.legend(handles=legend, fontsize=9)
plt.tight_layout()
plt.show()

### 3.2 Encoding Joint Actions as Integers

CQL encodes the joint action $(a_1, a_2, \ldots, a_n)$ as a single integer using **base-$d$ representation**:

$$\text{joint\_idx} = a_1 \cdot d^{n-1} + a_2 \cdot d^{n-2} + \ldots + a_n \cdot d^0$$

This turns a tuple of actions into one number that indexes into the Q-table.

In [ ]:
config_cql = {
    "alpha": 0.1, "gamma": 0.99, "epsilon": 1.0,
    "epsilon_decay": 0.995, "min_epsilon": 0.05,
    "action_dim": 5, "episodes": 300, "seed": 42,
}

cql = CQL(env, config_cql)

print("CQL internal structure:")
print(f"  Agent IDs        : {cql.agent_ids}")
print(f"  N agents         : {cql.n_agents}")
print(f"  Action dim       : {cql.action_dim}")
print(f"  Joint actions    : {cql.n_joint_actions}  ({cql.action_dim}^{cql.n_agents})")
print(f"  Q-table shape    : defaultdict → each key maps to np.zeros({cql.n_joint_actions})")

In [ ]:
# Demonstrate joint action encoding
print("Joint action encoding (2 agents, 5 actions each → 25 joint actions):\n")
print(f"{'predator_1 action':>20s} {'prey_1 action':>15s} {'joint index':>12s}")
print("-" * 50)

action_names = ["Right", "Up", "Left", "Down", "Noop"]
for a1 in range(cql.action_dim):
    for a2 in range(cql.action_dim):
        joint_idx = cql._joint_action_index({"predator_1": a1, "prey_1": a2})
        # Print only every 5th to keep output compact
        if a2 == 0:
            print(f"{action_names[a1]:>20s} {action_names[a2]:>15s} {joint_idx:>12d}")

print("  ...")
print(f"\nTotal: {cql.action_dim}^{cql.n_agents} = {cql.n_joint_actions} joint action indices")

---
## Part 4 — Marginalisation: From Joint to Per-Agent Decisions

The Q-table gives us Q-values over the full joint action space. But at execution time, each agent needs to decide on **its own action** independently.

**Marginalisation** averages out the other agents' contributions:

$$Q_i(\mathbf{s}, a_i) = \frac{1}{d^{n-1}} \sum_{a_{-i}} Q(\mathbf{s}, a_i, a_{-i})$$

Agent $i$ then acts greedily on $Q_i$:

$$a_i^* = \arg\max_{a_i} Q_i(\mathbf{s}, a_i)$$

This is equivalent to computing: *"what is the average Q-value for each of my actions, assuming the other agents act uniformly at random?"*

In [ ]:
# Demonstrate marginalisation on an untrained table (all zeros)
# Then on a manually seeded table so we can see the math clearly

obs, _ = env.reset(seed=42)
joint_state = cql._joint_state(obs)

# Manually insert Q-values so we can trace the math
fake_q = np.zeros(cql.n_joint_actions, dtype=np.float32)
# Set high Q-value for: predator_1=Right(0), prey_1=Left(2) → joint idx = 0*5+2 = 2
fake_q[2]  = 10.0   # predator goes Right, prey goes Left → capture likely
fake_q[7]  = 5.0    # predator goes Up(1), prey goes Left(2) → 1*5+2=7
cql.q_table[joint_state] = fake_q

print("Fake Q-table for this joint state (non-zero entries only):")
action_names = ["Right", "Up", "Left", "Down", "Noop"]
for idx, q in enumerate(fake_q):
    if q != 0:
        a1, a2 = divmod(idx, cql.action_dim)
        print(f"  [{action_names[a1]}, {action_names[a2]}] → idx={idx} → Q={q:.1f}")

# Marginalise for predator_1 (agent index 0)
q_tensor = fake_q.reshape((cql.action_dim, cql.action_dim))  # (pred_actions, prey_actions)
q_pred = q_tensor.mean(axis=1)  # average over prey's actions (axis=1)

print("\nMarginalised Q for predator_1 (averaged over all prey actions):")
for a, q in enumerate(q_pred):
    star = " ← greedy choice" if a == np.argmax(q_pred) else ""
    print(f"  {action_names[a]:>8s} ({a}): {q:.4f}{star}")

# Reset the table
cql.q_table.pop(joint_state, None)

---
## Part 5 — Training CQL

Same training loop as IQL, but one Q-table update per episode step instead of N.

In [ ]:
def train_cql_and_collect(env, config, n_episodes=300, log_every=50):
    """Train CQL and return the algorithm and episode metrics."""
    algo = CQL(env, config)
    
    rewards_per_episode = {aid: [] for aid in algo.agent_ids}
    episode_lengths     = []
    epsilon_history     = []
    capture_flags       = []
    q_table_sizes       = []
    
    for ep in range(n_episodes):
        obs, _ = env.reset()
        done   = False
        ep_reward = {aid: 0.0 for aid in algo.agent_ids}
        steps  = 0
        
        while not done:
            actions  = algo.select_actions(obs)
            result   = env.step(actions)
            next_obs = result["obs"]
            rewards  = result["reward"]
            done     = result["terminated"] or result["truncated"]
            
            joint_s      = algo._joint_state(obs)
            joint_s_next = algo._joint_state(next_obs)
            joint_a      = algo._joint_action_index(actions)
            
            # Central reward = sum of all agents' rewards
            central_r = 0.0
            for aid in algo.agent_ids:
                r = float(rewards[aid])
                ep_reward[aid] += r
                central_r += r
            
            # Centralized Bellman update on the shared Q-table
            q_curr     = algo.q_table[joint_s][joint_a]
            q_next_max = 0.0 if done else float(np.max(algo.q_table[joint_s_next]))
            td         = central_r + algo.gamma * q_next_max - q_curr
            algo.q_table[joint_s][joint_a] += algo.alpha * td
            
            obs    = next_obs
            steps += 1
        
        algo.epsilon = max(algo.min_epsilon, algo.epsilon * algo.epsilon_decay)
        
        for aid in algo.agent_ids:
            rewards_per_episode[aid].append(ep_reward[aid])
        episode_lengths.append(steps)
        epsilon_history.append(algo.epsilon)
        capture_flags.append(1 if result["terminated"] else 0)
        q_table_sizes.append(len(algo.q_table))
        
        if (ep + 1) % log_every == 0:
            avg_len  = np.mean(episode_lengths[-log_every:])
            cap_rate = np.mean(capture_flags[-log_every:]) * 100
            print(f"  Episode {ep+1:4d}/{n_episodes} | "
                  f"ε={algo.epsilon:.3f} | "
                  f"avg steps={avg_len:.1f} | "
                  f"capture rate={cap_rate:.1f}% | "
                  f"joint states={len(algo.q_table)}")
    
    metrics = {
        "rewards":      rewards_per_episode,
        "lengths":      episode_lengths,
        "epsilon":      epsilon_history,
        "capture_rate": capture_flags,
        "q_table_size": q_table_sizes,
    }
    return algo, metrics


print("Training CQL for 300 episodes on 5×5 grid (1 predator vs 1 prey)...")
env = make_env(n_predators=1, n_prey=1, size=5)
trained_cql, cql_metrics = train_cql_and_collect(env, config_cql, n_episodes=300, log_every=50)

### 5.1 CQL Training Curves

In [ ]:
def smooth(x, window=20):
    return np.convolve(x, np.ones(window)/window, mode="valid")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle("CQL Training — 5×5 grid, 1 predator vs 1 prey", fontsize=13)

# Predator reward
r = cql_metrics["rewards"]["predator_1"]
axes[0].plot(r, alpha=0.3, color="#e67e22")
axes[0].plot(smooth(r), color="#d35400", linewidth=2)
axes[0].set_title("Predator Reward")
axes[0].set_xlabel("Episode")
axes[0].set_ylabel("Total reward")
axes[0].grid(True, alpha=0.3)

# Episode length
l = cql_metrics["lengths"]
axes[1].plot(l, alpha=0.3, color="#9b59b6")
axes[1].plot(smooth(l), color="#8e44ad", linewidth=2)
axes[1].set_title("Episode Length")
axes[1].set_xlabel("Episode")
axes[1].set_ylabel("Steps")
axes[1].grid(True, alpha=0.3)

# Q-table growth
axes[2].plot(cql_metrics["q_table_size"], color="#1abc9c", linewidth=2)
axes[2].set_title("Joint State Table Size")
axes[2].set_xlabel("Episode")
axes[2].set_ylabel("Unique joint states seen")
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## Part 6 — IQL vs CQL Comparison

Now we train **both algorithms on the same environment** with the same config and compare their learning curves.

In [ ]:
def train_iql_and_collect(env, config, n_episodes=300):
    """Minimal IQL training loop returning episode metrics."""
    algo = IQL(env, config)
    lengths, captures, eps_hist = [], [], []
    
    for ep in range(n_episodes):
        obs, _ = env.reset()
        done, steps = False, 0
        
        while not done:
            actions  = algo.select_actions(obs)
            result   = env.step(actions)
            next_obs = result["obs"]
            done     = result["terminated"] or result["truncated"]
            
            for aid in algo.agent_ids:
                s = algo._encode_state(obs[aid])
                a = actions[aid]
                r = float(result["reward"][aid])
                s_next = algo._encode_state(next_obs[aid])
                q_curr = algo.q_tables[aid][s][a]
                q_next = 0.0 if done else float(np.max(algo.q_tables[aid][s_next]))
                algo.q_tables[aid][s][a] += algo.alpha * (r + algo.gamma * q_next - q_curr)
            
            obs, steps = next_obs, steps + 1
        
        algo.epsilon = max(algo.min_epsilon, algo.epsilon * algo.epsilon_decay)
        lengths.append(steps)
        captures.append(1 if result["terminated"] else 0)
        eps_hist.append(algo.epsilon)
    
    return algo, {"lengths": lengths, "capture_rate": captures, "epsilon": eps_hist}


N_EPISODES = 300
config_shared = {
    "alpha": 0.1, "gamma": 0.99, "epsilon": 1.0,
    "epsilon_decay": 0.995, "min_epsilon": 0.05,
    "action_dim": 5, "episodes": N_EPISODES, "seed": 42,
}

env_1v1 = make_env(n_predators=1, n_prey=1, size=5, seed=42)

print("Training IQL...")
_, iql_m = train_iql_and_collect(env_1v1, config_shared, n_episodes=N_EPISODES)

print("Training CQL...")
_, cql_m = train_cql_and_collect(env_1v1, config_shared, n_episodes=N_EPISODES, log_every=N_EPISODES)

print("Both trained.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle("IQL vs CQL — 5×5 grid, 1 predator vs 1 prey, 300 episodes", fontsize=13)

# Episode length
ax = axes[0]
ax.plot(smooth(iql_m["lengths"]), color="#e74c3c", linewidth=2, label="IQL")
ax.plot(smooth(cql_m["lengths"]), color="#3498db", linewidth=2, label="CQL")
ax.set_title("Episode Length (↓ = captures happen faster)")
ax.set_xlabel("Episode")
ax.set_ylabel("Steps (smoothed, window=20)")
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

# Capture rate (rolling 20-episode window)
ax = axes[1]
iql_cap = smooth(iql_m["capture_rate"]) * 100
cql_cap = smooth(cql_m["capture_rate"]) * 100
ax.plot(iql_cap, color="#e74c3c", linewidth=2, label="IQL")
ax.plot(cql_cap, color="#3498db", linewidth=2, label="CQL")
ax.set_title("Capture Rate (↑ = predator captures more often)")
ax.set_xlabel("Episode")
ax.set_ylabel("% episodes with capture (window=20)")
ax.set_ylim(0, 105)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Final 50-episode summary:")
for name, m in [("IQL", iql_m), ("CQL", cql_m)]:
    avg_len  = np.mean(m["lengths"][-50:])
    cap_rate = np.mean(m["capture_rate"][-50:]) * 100
    print(f"  {name}: avg length={avg_len:.1f} steps, capture rate={cap_rate:.1f}%")

---
## Part 7 — The Scaling Limit in Practice

Let's verify that CQL's joint state table does indeed grow much faster as we add agents.

In [ ]:
# Train CQL briefly (100 episodes) for different agent counts
# and measure how large the joint Q-table becomes

configs_to_test = [
    {"n_pred": 1, "n_prey": 1, "size": 5},
    {"n_pred": 1, "n_prey": 2, "size": 6},
    {"n_pred": 2, "n_prey": 1, "size": 6},
    {"n_pred": 2, "n_prey": 2, "size": 7},
]

N_QUICK = 50  # short run just to see table growth

print(f"Tracking joint Q-table size after {N_QUICK} episodes:")
print(f"{'Config':>20s} {'Agents':>8s} {'Joint actions':>15s} {'States seen':>13s}")
print("-" * 60)

for cfg in configs_to_test:
    e = make_env(n_predators=cfg["n_pred"], n_prey=cfg["n_prey"], size=cfg["size"])
    n = cfg["n_pred"] + cfg["n_prey"]
    quick_config = {**config_shared, "episodes": N_QUICK}
    _, m = train_cql_and_collect(e, quick_config, n_episodes=N_QUICK, log_every=N_QUICK)
    joint_actions = 5 ** n
    final_size = m["q_table_size"][-1]
    label = f"{cfg['n_pred']}P vs {cfg['n_prey']}R (size={cfg['size']})"
    print(f"{label:>20s} {n:>8d} {joint_actions:>15,d} {final_size:>13,d}")

---
## Part 8 — When to Use What

### Summary: IQL vs CQL Trade-offs

| Property | IQL | CQL |
|----------|-----|-----|
| Q-tables | One per agent | One shared |
| State | Per-agent observation | Joint observation of all agents |
| Action selection | Per-agent argmax | Marginalised argmax |
| Reward signal | Per-agent reward | Sum of all rewards |
| Stationarity | ❌ Non-stationary (other agents learn) | ✅ Stationary (reasons about all jointly) |
| Memory scaling | Linear in agents | Exponential in agents |
| Convergence guarantee | None in MARL | Theoretically sound for cooperative tasks |
| Practical limit | Any scale | ≤ ~4 agents, small grids |

### Rule of Thumb

- **Use IQL** when you have many agents, a large state space, or want a simple scalable baseline.
- **Use CQL** when you have ≤ 4 agents, a small grid, and want a theoretically grounded upper-bound comparison.
- **Neither scales** to large multi-agent problems — that's where deep RL (DQN, PPO) and game-theoretic methods (Nash Q-learning) come in.

In [ ]:
# Final sanity check: save and reload CQL
save_path = str(PROJECT_ROOT / "notebooks" / "cql_demo_checkpoint.pkl")
trained_cql.save(save_path)

env_fresh = make_env()
loaded_cql = CQL.load(env_fresh, config_cql, save_path)

obs, _ = env_fresh.reset(seed=99)
trained_cql.epsilon = 0.0
loaded_cql.epsilon  = 0.0

a_trained = trained_cql.select_actions(obs)
a_loaded  = loaded_cql.select_actions(obs)

print(f"Actions trained : {a_trained}")
print(f"Actions loaded  : {a_loaded}")
print(f"Match           : {a_trained == a_loaded}")

---
## Summary

In this notebook you:

| Step | What happened |
|------|---------------|
| Understood non-stationarity | Why IQL has no MARL convergence guarantee |
| Saw exponential scaling | Joint action space grows as $d^n$ |
| Traced joint action encoding | How a tuple of actions becomes one integer |
| Traced marginalisation | How the joint Q-table produces per-agent decisions |
| Trained CQL | With live metrics on joint state table growth |
| Compared IQL vs CQL | Same environment, same config — side-by-side curves |
| Hit the scaling wall | Verified that CQL grows impractically for 4+ agents |

## Exercises

1. **Adversarial setting:** Run both IQL and CQL. Does CQL's cooperative reward (sum of all agents) work well when predator and prey have **opposite** goals? What happens to the prey's Q-table under CQL?

2. **Memory count:** After training CQL for 300 episodes on a 6×6 grid with 2 predators and 2 prey, how many joint states are in the table? How much RAM does that consume?

3. **Marginalisation choice:** The code uses `.mean()` over other agents' Q-values. Change it to `.max()`. Does training behavior change? Which is more theoretically justified?

4. **Central vs. local reward:** Modify the `train_cql_and_collect` loop to use only the predator's reward as the central signal (remove prey reward). Does CQL still converge? Why or why not?

5. **State space comparison:** Plot the number of unique states seen by IQL (per agent) vs CQL (joint) over training. Why does the joint state space grow so much faster?

---
*Next: `03_nash_q_demo.ipynb` — Game-theoretic Q-learning and Nash equilibria (coming in Week 7)*